## Calling libraries

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import difflib

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

## Loading data

In [ ]:
RNG_SEED = 42

train_app = pd.read_csv('data/applications_train.csv')
test_app = pd.read_csv('data/applications_test.csv')
jobs = pd.read_csv('data/jobs.csv')
candidates = pd.read_csv('data/candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

train_job_ids = set(train_df['job_id'].unique())

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

def _truthy(s):
    return s.fillna(0).astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 't'])

Part 1: Data Retrieval and Base Variable Preparation

In this part of the pipeline, the initial infrastructure for data processing is created. The main goal of this step is to aggregate relational data and define mapping structures for use in the feature engineering stages. The steps performed in this section are as follows:

Setting the Seed: Initializing the RNG_SEED to ensure repeatability of results in the training and validation stages of the model.

Data Merging: Reading independent files (candidates, jobs, and requests) and merging them using the Left Join operation on the job_id and candidate_id keys to form comprehensive training and testing datasets.

Extracting observed job IDs: Creating a unique set of job IDs in the training set to handle Cold-Start scenarios during the final prediction.

Define ordinal encoding: Create standard dictionaries to convert ordinal text data such as English language level and educational level to continuous numeric values.

Currency normalization: Define currency conversion rates to normalize salary columns to a common currency (dollars).

Develop helper functions: Implement the truthy_ function to clean up and consolidate Boolean values ​​recorded in various text formats.

## Initial preprocessing and calculation of base distances

In [3]:
_exp_sal_med = None
_sal_min_med = None
_sal_max_med = None

for i, df in enumerate([train_df, test_df]):
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)

    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)

    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates

    if i == 0: 
        _exp_sal_med = df['expected_salary'].median()
        _sal_min_med = df['salary_min_usd'].median()
        _sal_max_med = df['salary_max_usd'].median()

    df['expected_salary'] = df['expected_salary'].fillna(_exp_sal_med)
    df['salary_min_usd'] = df['salary_min_usd'].fillna(_sal_min_med)
    df['salary_max_usd'] = df['salary_max_usd'].fillna(_sal_max_med)

    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']

    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) /
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(
            None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

Part 2: Base Preprocessing & Feature Engineering

In this part, the raw data is cleaned and basic and logical features are extracted from it. One of the key principles observed in this step is to prevent data leakage during imputation. The most important operations in this part are:

Leakage-Free Imputation: Calculating the median values ​​for paid and requested salaries only on the training dataset (Train) and generalizing it to the test dataset (Test).

Ordinal Encoding: Applying the defined mappings to convert the level of English proficiency and educational levels into numerical values.

Time Feature Engineering: Calculating the days_to_apply variable to check the speed of the job seeker in sending a resume after publishing a job advertisement.

Currency Normalization: Converting the minimum and maximum salary values ​​of jobs to dollars based on exchange rates to create a uniform scale.

Calculating logical gaps (Gap Analysis): Extracting key features such as salary_gap (the gap between the requested salary and the organization's budget) and experience_gap (the gap between the candidate's experience and the minimum required experience).

Matching Metrics: * Calculating skill_match_ratio based on the similarity of the candidate's skills and job requirements.

Measuring the title similarity (title_similarity) between the candidate's current job and the target job position using String Matching techniques.

## Advanced Feature Engineering

In [4]:
def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}

def build_advanced_features(df, fitted_idf=None, fitted_company_freq=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal,
                                          np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    if fitted_company_freq is None:
        freq = Counter()
        for s in prev:
            freq.update(s)
    else:
        freq = fitted_company_freq
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)

    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]

    g = df.groupby('job_id')
    df["expgrank_x_salgrank"] = g["experience_gap"].rank(pct=True) * g["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * g["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    rs = g["skill_match_idf"].rank(ascending=False, method="min")
    re = g["exp_band_distance"].rank(ascending=True, method="min")
    df["consensus_rr"] = 1.0 / (rs + 1.0) + 1.0 / (re + 1.0)
    df["rank_spread"] = (rs - re).abs()

    df["skill_z"] = g["skill_match_idf"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
    df["skill_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]
    df["idf_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]

    same_loc = (df["candidate_location"].fillna("").astype(str).str.lower() ==
                df["job_location"].fillna("").astype(str).str.lower()).astype(int)
    remote_ok = _truthy(df["remote_allowed"]) if "remote_allowed" in df.columns else pd.Series(False, index=df.index)
    relocate_ok = _truthy(df["willing_to_relocate"]) if "willing_to_relocate" in df.columns else pd.Series(False, index=df.index)
    df["loc_same"] = same_loc
    df["loc_compatible"] = ((same_loc == 1) | remote_ok | relocate_ok).astype(int)
    df["loc_friction"] = ((1 - same_loc) * (~remote_ok).astype(int)).astype(int)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity',
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband",
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap",
        "consensus_rr", "skill_gap_to_best", "loc_compatible",
    ]
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    return df, idf, freq

train_df, train_idf, train_company_freq = build_advanced_features(train_df)
test_df, _, _ = build_advanced_features(test_df, fitted_idf=train_idf, fitted_company_freq=train_company_freq)

Third layer: Advanced Feature Engineering & Group Ranking

This is the core of knowledge extraction for Learning-to-Rank (LTR) algorithms. The goal here is to move from absolute candidate features to “relative and competitive features” in the context of a specific job position. Techniques and innovations implemented in this section include:

Natural Language Processing (NLP Tokenization): Parsing and standardizing skills, certifications, and job titles into a set of tokens for performing set operations such as intersection and calculating Jaccard similarity index.

Semantic Weighting (TF-IDF Matching): Calculating the information value (IDF) of each skill based on its prevalence in the overall market requirements. Rare and specialized skills receive a much higher weight than generic skills to better identify key matches (skill_match_idf). Note: To prevent data leakage, IDF values ​​are fitted exclusively to the training set and then applied to the test set.

Band & Friction Analysis: Calculates continuous and discrete penalties for candidates who fall outside the employer's desired ranges (in terms of experience, salary budget, and location/teleworking permission).

Cross-Features: Generates complex composite variables such as match_per_salovershoot, which simulates the concept of a candidate's "return on investment (ROI)" by dividing the skill level by the requested salary surplus.

Contextual Ranking: By grouping data by job_id, competitive features such as Z-Score and Percentile Rank are calculated for each candidate compared to other competitors in the same job posting. This helps the model identify the best candidate in a given pool without relying on absolute metrics.

Reciprocal Rank Consensus: Uses the harmonic combination of ranks (consensus_rr) to find candidates who are simultaneously the highest in skill and the lowest in distance from the employer.

## Building features based on text vectors (TF-IDF & SVD)

In [5]:
def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]

    med = train_df['expected_salary'].median()
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (med + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (med + 1)

    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)

        job_tr = tfidf.transform(train_df[col_job].fillna(""))
        cand_tr = tfidf.transform(train_df[col_cand].fillna(""))
        job_te = tfidf.transform(test_df[col_job].fillna(""))
        cand_te = tfidf.transform(test_df[col_cand].fillna(""))

        svd = TruncatedSVD(n_components=10, random_state=RNG_SEED)
        job_tr_s = svd.fit_transform(job_tr)
        cand_tr_s = svd.transform(cand_tr)
        job_te_s = svd.transform(job_te)
        cand_te_s = svd.transform(cand_te)

        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_tr_s, cand_tr_s)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_te_s, cand_te_s)

        prod_tr = job_tr_s * cand_tr_s
        prod_te = job_te_s * cand_te_s
        train_df[f'{col_job}_svd_dotmax'] = prod_tr.max(axis=1)
        train_df[f'{col_job}_svd_dotsum'] = prod_tr.sum(axis=1)
        test_df[f'{col_job}_svd_dotmax'] = prod_te.max(axis=1)
        test_df[f'{col_job}_svd_dotsum'] = prod_te.sum(axis=1)

    return train_df, test_df

train_df, test_df = add_lsa_and_global_features(train_df, test_df)

Part 4: Text Semantic Analysis (LSA) and Global Features

In this step, Natural Language Processing (NLP) is applied to the text fields in a more advanced way so that the model goes beyond simple exact match to understand latent semantics. Features independent of the current job context are also generated for each candidate. The steps in this part are:

Global Profiling: Extracting the profile_strength index as a heuristic measure of the raw strength of the resume, and the global_salary_ratio index to understand the candidate’s financial position relative to the median of the entire market.

Text Vectorization (TF-IDF Vectorization): Converting text fields (combinations of skills and job titles) into a vector space using the word frequency distribution (single-word and two-word) with a maximum of 5000 features. To prevent data leakage, the vocabulary is fitted exclusively to the training set.

Truncated SVD / LSA: Compresses the TF-IDF redundancy matrix into a dense 10-dimensional space. This technique models word co-occurrence and extracts the main concepts and topics of the texts.

Cosine Similarity: Calculates the cosine similarity between the compressed job ad vector and the candidate vector in the new semantic space.

Extraction of Sign-Invariant Dot Summaries: In order to avoid the inherent instabilities of SVD vectors (such as sign rotation) in tree-based models, instead of using raw SVD components, aggregate features including maximum and sum of element-wise dot products are used. These values ​​provide a strong signal of overlap between key concepts.

## Target Encoding Setup and Settings

In [6]:
HIGH_CARD_COLS = ['current_title', 'job_location', 'university', 'industry']
TE_K = 20
TE_NOISE = 0.01
TE_MIN_COUNT = 1

def _norm_cat(series):
    return series.fillna("MISSING").astype(str).str.lower().str.strip()

for c in HIGH_CARD_COLS:
    train_df[c + "_normcat"] = _norm_cat(train_df[c])
    test_df[c + "_normcat"] = _norm_cat(test_df[c])

train_df["_nl_full"] = train_df.groupby("job_id")["relevance_label"].transform(
    lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
_global_gmean = train_df["_nl_full"].mean()

test_te_maps = {}
test_freq_maps = {}
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    test_te_maps[c] = smooth
    test_freq_maps[c] = train_df[c + "_normcat"].value_counts()

def fit_te_on_fold(fold_train_df, cat_col, gmean):
    agg = fold_train_df.groupby(cat_col + "_normcat")["_nl_fold"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    return (agg["mean"] * agg["count"] + gmean * TE_K) / (agg["count"] + TE_K)

Section 5: Target Encoding with Smoothing In this section, one of the most powerful feature engineering techniques is implemented for managing high-cardinality categorical variables such as city names, universities, and job titles. Using traditional methods such as One-Hot Encoding on these variables leads to the Curse of Dimensionality phenomenon; therefore, the Smoothed Target Encoding approach was used. The salient features of this implementation are: Normalization: Preprocessing of the categorical texts by converting them to lowercase and removing extra spaces to prevent the creation of false duplicate categories. Contextual Target Formulation: Instead of using the raw label, the candidate's percentage rank in their job group (_nl_full) is used as the target variable. This makes the encoded values ​​consistent with the nature of Learning-to-Rank algorithms. Bayesian Smoothing: To prevent overfitting on rare categories, a tuning parameter of $K=20$ is used. This formula combines the mean of each category with the global mean, so that rare categories tend towards the global mean and frequent categories maintain their weight. Frequency Encoding: In addition to Target Encoding, the number of occurrences of each category is also extracted as a signal of popularity. Leakage-Free Design: Development of a helper function fit_te_on_fold to dynamically apply Target Encoding in cross-validation (CV) loops. This design ensures that no information is leaked from the validation set targets to the training set during model training (Data Leakage Prevention).

## Feature Selection - Null Importance

In [7]:
cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed',
    'company_size', 'industry', 'current_title', 'skills', 'education_level',
    'university', 'previous_companies', 'certifications', 'english_proficiency',
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label',
    '_nl_full', '_nl_fold',
] + [c + "_normcat" for c in HIGH_CARD_COLS]

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = np.nan
    train_df[f"{c}_freq"] = 0.0
    test_df[f"{c}_te"] = test_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    test_df[f"{c}_freq"] = test_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

candidate_features = [
    c for c in train_df.columns
    if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]

def null_importance_pruner(X, y, groups, features, n_runs=15, pct=75, seed=0):
    sort_idx = np.argsort(groups, kind='stable')
    Xs = X.iloc[sort_idx].reset_index(drop=True)
    ys = y[sort_idx].copy()
    _, counts = np.unique(groups[sort_idx], return_counts=True)

    base = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                          random_state=seed, n_jobs=-1)
    base.fit(Xs, ys, group=counts)
    actual = base.booster_.feature_importance("gain")

    null = np.zeros((n_runs, len(features)))
    for r in range(n_runs):
        yp = ys.copy()
        start = 0
        rs = np.random.RandomState(1000 + r)
        for cnt in counts:
            block = yp[start:start + cnt].copy()
            rs.shuffle(block)
            yp[start:start + cnt] = block
            start += cnt
        m = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                           random_state=r, n_jobs=-1)
        m.fit(Xs, yp, group=counts)
        null[r] = m.booster_.feature_importance("gain")

    thresh = np.percentile(null, pct, axis=0)
    score = np.log1p(actual) - np.log1p(thresh)
    keep = [f for f, s in zip(features, score) if s > 0]
    if len(keep) < 15:
        order = np.argsort(actual)[::-1]
        keep = [features[i] for i in order[:50]]
    return keep

te_cols = [f"{c}_te" for c in HIGH_CARD_COLS] + [f"{c}_freq" for c in HIGH_CARD_COLS]
static_features = [c for c in candidate_features if c not in te_cols]

kept_static = null_importance_pruner(
    train_df[static_features],
    train_df['relevance_label'].values,
    train_df['job_id'].values,
    static_features,
)
final_features = kept_static + te_cols
print(f"[pruner] kept {len(kept_static)} static + {len(te_cols)} TE = {len(final_features)} features")

/tmp/ipykernel_211897/2748792334.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f"{c}_te"] = np.nan
/tmp/ipykernel_211897/2748792334.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f"{c}_freq"] = 0.0
/tmp/ipykernel_211897/2748792334.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fr

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.069780 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16068
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 87
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16068
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 87
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061035 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16068
[LightGBM] [Info] Number of d

Part 6: Intelligent Feature Selection and Noise Removal (Feature Selection via Null Importance)

In this step, the feature space is cleaned and variables with no predictive value (noise) are removed. Using all the features generated in the previous steps can lead to the phenomenon of Overfitting in tree algorithms. The approach adopted in this section is as follows:

Initial Filtering (Data Type Filtering): Definitely removing all unique identifiers (IDs), date variables, raw texts and features containing information leakage (Leakage Columns). In this step, only numeric variables (float and int) are allowed to enter the validation phase.

Null Importance Algorithm: Instead of using traditional Feature Importance methods that are prone to bias towards High-Cardinality features, a noise-based approach is used:

Baseline Evaluation: Calculating the actual importance of each feature (Actual Gain) by an initial LightGBM model.

Generate Random Distribution: Shuffling labels within each job_id over multiple independent iterations (n_runs=15) and recording the random importance of each feature (Null Gain).

Statistical Pruning: Logarithmic comparison between the actual importance and the 75th percentile of the noise distribution. Only features whose signal is significantly stronger than the random noise are retained.

Fallback Safety: Embedding a control condition so that in case of excessive pruning of the algorithm (reducing the features to less than 15), the system automatically returns the top 50 features based on Actual Gain to prevent Underfitting.

Dynamic Features Protection: Features generated by Target Encoding are excluded from the pruning process due to their fold-dependent nature and are added directly to the final feature list.

## Cross-validation and training of the main models

In [8]:
train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

def ndcg_by_group(y_true, y_pred, groups, k=10):
    d = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k)
         for _, gr in d.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'eval_at': [10],
    'lambdarank_truncation_level': 12,   
    'n_estimators': 3000,
    'learning_rate': 0.02,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 30,
    'min_split_gain': 0.0,               
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,           
    'reg_alpha': 0.5,
    'reg_lambda': 5.0,
    'label_gain': [0, 1, 3, 7, 15],
    'random_state': RNG_SEED,
    'n_jobs': -1,
    'verbosity': -1,
}

cat_params = dict(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=RNG_SEED,
    verbose=0,
)

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(5):
    val_jobs = folds[i]
    tr_jobs = np.concatenate([folds[j] for j in range(5) if j != i])
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    cv_splits.append((tr_idx, val_idx))

oof_lgb = np.full(len(train_df), np.nan)
oof_cat = np.full(len(train_df), np.nan)
lgb_best_iters, cat_best_iters = [], []
rng = np.random.default_rng(RNG_SEED)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    fold_tr = train_df.iloc[tr_idx].copy()
    fold_tr["_nl_fold"] = fold_tr.groupby("job_id")["relevance_label"].transform(
        lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = fold_tr["_nl_fold"].mean()

    for c in HIGH_CARD_COLS:
        smap = fit_te_on_fold(fold_tr, c, gmean)
        val_vals = train_df.iloc[val_idx][c + "_normcat"].map(smap).fillna(gmean).values
        train_df.loc[train_df.index[val_idx], f"{c}_te"] = val_vals
        tr_vals = train_df.iloc[tr_idx][c + "_normcat"].map(smap).fillna(gmean).values
        tr_vals = tr_vals * (1 + rng.normal(0, TE_NOISE, size=len(tr_vals)))
        train_df.loc[train_df.index[tr_idx], f"{c}_te"] = tr_vals
        freq_map = fold_tr[c + "_normcat"].value_counts()
        train_df.loc[train_df.index[tr_idx], f"{c}_freq"] = \
            train_df.iloc[tr_idx][c + "_normcat"].map(freq_map).fillna(0).values
        train_df.loc[train_df.index[val_idx], f"{c}_freq"] = \
            train_df.iloc[val_idx][c + "_normcat"].map(freq_map).fillna(0).values

    X = train_df[final_features]

    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr = X.iloc[tr_idx[tr_order]]
    ytr = y[tr_idx[tr_order]]
    gtr = groups[tr_idx[tr_order]]

    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva = X.iloc[val_idx[va_order]]
    yva = y[val_idx[va_order]]
    gva = groups[val_idx[va_order]]

    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)

    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr,
           eval_set=[(Xva, yva)], eval_group=[c_va],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_best_iters.append(lm.best_iteration_ or lgb_params['n_estimators'])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])

    cm = CatBoostRanker(**cat_params, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    cat_best_iters.append(cm.get_best_iteration() or cat_params['iterations'])
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

    f_lgb = ndcg_by_group(y[val_idx], oof_lgb[val_idx], groups[val_idx])
    f_cat = ndcg_by_group(y[val_idx], oof_cat[val_idx], groups[val_idx])
    print(f"[fold {fold}] NDCG@10  lgb={f_lgb:.4f}  cat={f_cat:.4f}  "
          f"(lgb_iter={lgb_best_iters[-1]}, cat_iter={cat_best_iters[-1]})")

/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[fold 0] NDCG@10  lgb=0.8927  cat=0.8906  (lgb_iter=830, cat_iter=634)


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[fold 1] NDCG@10  lgb=0.8824  cat=0.8869  (lgb_iter=915, cat_iter=1202)


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[fold 2] NDCG@10  lgb=0.8823  cat=0.8881  (lgb_iter=841, cat_iter=1089)


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[fold 3] NDCG@10  lgb=0.8932  cat=0.8986  (lgb_iter=828, cat_iter=1210)


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[fold 4] NDCG@10  lgb=0.8749  cat=0.8799  (lgb_iter=671, cat_iter=791)


Part 7: Cross-Validation, Ranker Training, and Out-of-the-Fold Predictions

This part is the main machine learning phase of the pipeline. In this step, two powerful Gradient Boosting algorithms, LightGBM (with LambdaRank objective function) and CatBoost (with YetiRank objective function), are used to solve the Learning-to-Rank problem. The key strategies and techniques implemented in this part are:

Group/Query Structuring: Accurately sort the data based on job_id to synchronize with the input structure of the Ranker algorithms, so that tree comparisons are made exclusively within each job posting.

Strict Evaluation (NDCG@10 Evaluation): Develop custom functions to calculate the Normalized Discounted Cumulative Gain on the top 10 results of each group. This metric forces the model to place the best candidates in the highest possible ranks (Top Ranks).

Time-Based CV Splits: Splitting job positions into 5 folds based on the maximum CV submission date. This approach ensures the robustness of the model against temporal changes in the job market.

Fold-Specific Target Encoding with Noise: To prevent data leakage, Smoothed Target Encoding values ​​are calculated dynamically within each fold, using only the training data of that fold. Also, injecting Gaussian Noise into the training data increases the robustness of the model and prevents overfitting on specific categories.

Model Training & Early Stopping: Feed data to two tree models with optimized settings (such as tree depth control, L2 penalty, and low learning rate). The Early Stopping mechanism is set to 50 iterations to extract the optimal learning stopping point (Best Iteration).

OOF Results Aggregation: Extract and store the Out-Of-Fold predictions for both models, which will be the main basis for the next stage (Ensembling) to find the optimal combination weights

## Combining results and extracting optimal weights (Ensemble Optimization)

In [9]:
valid = ~np.isnan(oof_lgb)
r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)

print(f"\n[OOF] lgb={ndcg_by_group(y[valid], oof_lgb[valid], groups[valid]):.4f}  "
      f"cat={ndcg_by_group(y[valid], oof_cat[valid], groups[valid]):.4f}")

def power_blend(rl, rc, w, p):
    return w * (rl ** p) + (1 - w) * (rc ** p)

def best_blend(mask):
    bs, bw, bp = -1.0, 0.5, 1.0
    for p in (1.0, 1.5, 2.0):
        for w in np.linspace(0, 1, 41):
            sc = ndcg_by_group(y[mask], power_blend(r_lgb, r_cat, w, p)[mask], groups[mask])
            if sc > bs:
                bs, bw, bp = sc, w, p
    return bw, bp, bs

w_all, p_all, s_all = best_blend(valid)
print(f"[blend] global  w={w_all:.3f} p={p_all}  NDCG@10={s_all:.4f}")

w_seen, p_seen = w_all, p_all
w_cold, p_cold = max(0.0, w_all - 0.15), p_all


[OOF] lgb=0.8851  cat=0.8888
[blend] global  w=0.100 p=1.0  NDCG@10=0.8888


Part 8: Ensemble Learning and Blend Optimization In this step, the predictions from LightGBM and CatBoost are combined to overcome the individual weaknesses of the models and utilize their learning diversity. The architecture and strategies implemented in this part include: Normalization of predictions (Rank Transformation): Due to the differences in the distribution and scale of the raw output scores of different models, the out-of-fold predictions are transformed into within-group percentile ranks. This technique makes the combination of models scale-invariant. Standalone OOF Evaluation: Calculate and report the NDCG@10 score for each model separately to create a baseline before combining. Power Blending: Use the nonlinear blending function $Score = w \cdot Rank_{LGBM}^p + (1-w) \cdot Rank_{CAT}^p$. Applying a power of $p \ge 1$ results in an exponential focus on top-ranked candidates, which is highly consistent with the nature of the NDCG metric. Grid Search Optimization: Develop an exact search mechanism to find the most optimal combined weight ($w$) and power ($p$) by maximizing the NDCG score on the entire valid OOF data. Dynamic Cold-Start Strategy (Piecewise Weight Assignment): Implement a heuristic approach for completely new job positions (Unseen/Cold-Start). Since the CatBoost algorithm generally performs more robustly when faced with unseen groups, the weight assigned to it for this category of jobs is increased with a 15% bias.

## Final training on the entire data and generation of the submit file

In [10]:
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    train_df[f"{c}_te"] = train_df[c + "_normcat"].map(smooth).fillna(_global_gmean).values
    train_df[f"{c}_freq"] = train_df[c + "_normcat"].map(
        train_df[c + "_normcat"].value_counts()).fillna(0).values

X = train_df[final_features]
_, c_full = np.unique(groups, return_counts=True)
sort_full = np.argsort(groups, kind="stable")

lgb_final_iter = int(np.median(lgb_best_iters))
cat_final_iter = int(np.median(cat_best_iters))
print(f"[final] lgb_iter={lgb_final_iter}  cat_iter={cat_final_iter}")

final_lgb_params = {**lgb_params, 'n_estimators': lgb_final_iter}
final_lgb = lgb.LGBMRanker(**final_lgb_params)
final_lgb.fit(X.iloc[sort_full], y[sort_full], group=c_full)

final_cat_params = {**cat_params, 'iterations': cat_final_iter}
final_cat = CatBoostRanker(**final_cat_params)
final_cat.fit(Pool(X.iloc[sort_full], y[sort_full], group_id=groups[sort_full]))

X_test = test_df[final_features]
test_groups = test_df['job_id'].values

rt_lgb = group_rank(final_lgb.predict(X_test), test_groups)
rt_cat = group_rank(final_cat.predict(X_test), test_groups)

is_cold = ~test_df['job_id'].isin(train_job_ids).values
final_blend = np.empty(len(test_df))
final_blend[~is_cold] = power_blend(rt_lgb, rt_cat, w_seen, p_seen)[~is_cold]
final_blend[is_cold] = power_blend(rt_lgb, rt_cat, w_cold, p_cold)[is_cold]

print(f"[test] cold-start rows: {is_cold.sum()} / {len(is_cold)} "
      f"({100*is_cold.mean():.1f}%)")

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('sub_file.csv', index=False)


[final] lgb_iter=830  cat_iter=1089


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[test] cold-start rows: 48392 / 52700 (91.8%)


Part 9: Training the Final Models on the Full Data and Generating Output (Final Training & Inference)

This is the final phase (Production/Submission Phase) of the pipeline, where the knowledge extracted from the validation (CV) stages is used to train the final models and make predictions on the test set. The architecture of this part includes the following steps:

Generate Global Target Encoding: Recalculate and apply Smoothed Target Encoding to 100% of the training data set. Since the evaluation phase is over, using the full data will produce the most accurate TE Maps to transfer to the test set.

Optimal Stopping Retrieval: To avoid overfitting or underfitting in the final training, the median of the best iterations obtained in the 5 validation folds was used as the number of final trees (n_estimators) for the LightGBM and CatBoost models.

Full-Data Training: Fit the final Ranker models on the entire training data, observing the group_id structure and exact sorting.

Rank-Based Inference: Extract the model predictions for the test set and convert the raw scores to within-group percentile ranks for standardization before blending.

Segmented and Smart Ensembling: Apply a Cold-Start strategy during blending. Job IDs (job_id) are divided into two categories: "Seen" and "New" (Cold). Then, for new job positions, the combined weights are automatically shifted in favor of the more robust algorithm (CatBoost).

Standard output generation: Create the final dataframe, sort by application_id and save in CSV format for sending to the arbitration system. (End of pipeline).